# Experiment: SpectralBio Review Recovery MSH2 ESM-1v Audit (H100)

This notebook runs one high-ROI stronger-baseline audit to test whether the covariance signal helps on a non-flagship gene.

## Why this notebook exists
- The review already accepts that BRCA2 is positive, but treats it as a possibly isolated case.
- TP53 is not the right rescue target because the fixed `alpha=0.55` augmentation hurts against the ESM-1v ensemble.
- MSH2 is the best one-hour H100 target because it has decent support and fits in `ESM-1v` full-sequence mode (`934 aa`), which is much cheaper than local-window genes such as BRCA2 or NSD1.
- This version reuses the old support-panel artifacts when available, so it avoids re-scanning ClinVar and redoing the reference 150M scores.

## Success criteria
- Primary: `delta_augmented_vs_esm1v > 0` and paired bootstrap `ci_2p5 > 0`.
- Secondary: permutation audit shows the observed gain is stronger than shuffled-covariance controls.
- Writing decision:
  - strong positive -> update only `abstract` and `content`
  - ambiguous or negative -> keep the central claim conservative and only clean the narrative


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioReviewRecoveryMSH2')
SAVE_RESULTS_ZIP = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print(f'Drive mounted at {DRIVE_MOUNT_POINT}')
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')

print('ACABEI , SIGA PARA O PR?XIMO')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('SPECTRALBIO_REPO_URL', 'https://github.com/DaviBonetto/SpectralBio.git')
REPO_BRANCH = os.environ.get('SPECTRALBIO_REPO_BRANCH', 'codex/claw4s-rebuild')
DEFAULT_REPO_ROOT = Path('/content/Stanford-Claw4s')
ENV_REPO_ROOT = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()

def _looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()

candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])

REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_review_recovery_msh2_h100_complete'

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])

src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

if not BOOTSTRAP_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'], check=False)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'accelerate>=1.0.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Continuing in the same runtime.')
else:
    print('Bootstrap marker found; skipping reinstall.')

print('REPO_ROOT =', REPO_ROOT)
print('Python =', sys.version.split()[0])
print('ACABEI , SIGA PARA O PR?XIMO')


## Plan

1. Reuse the frozen support-panel artifacts if they exist in the repo.
2. Reuse the existing MSH2 reference `ESM2-150M` scores if they exist.
3. Run only the 5-model `ESM-1v` ensemble for `MSH2`.
4. Run the matched permutation audit and package the outputs.

This is much safer for the one-hour budget than rebuilding the whole support scan from scratch.


In [ ]:
import json
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, display

from spectralbio.supplementary.final_accept_hardening import (
    AcceptHardeningConfig,
    build_support_ranked_panel_manifest,
    create_accept_hardening_paths,
    run_esm1v_augmentation_suite,
    run_esm1v_permutation_audit,
)

TARGET_GENE = 'MSH2'
ESM1V_MODEL_IDS = tuple(f'facebook/esm1v_t33_650M_UR90S_{i}' for i in range(1, 6))
PERMUTATION_REPLICATES = 1000
RUN_3B_SHADOW = False
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'review_recovery_msh2_h100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    large_model_gene_limit=1,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)

def _resolve_repo_path(raw: str) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path
    content_prefix = '/content/Stanford-Claw4s/'
    if raw.startswith(content_prefix):
        candidate = REPO_ROOT / raw[len(content_prefix):]
        if candidate.exists():
            return candidate
    return raw_path

candidate_panel_roots = [
    REPO_ROOT / 'notebooks' / 'Final Results 1 e 2' / 'spectralbio_final_accept_part1_bundle',
    REPO_ROOT / 'notebooks' / 'Results Finais' / 'final_accept_part6_panel25_brca1_failure_results' / 'part6_panel25_brca1_failure' / 'panel25',
]
panel_root = next((root for root in candidate_panel_roots if (root / 'support_ranked_panel_manifest.json').exists()), None)

if panel_root is not None:
    panel_manifest = json.loads((panel_root / 'support_ranked_panel_manifest.json').read_text(encoding='utf-8'))
    for gene_name, gene_info in panel_manifest.get('genes', {}).items():
        gene_info['sequence_path'] = str(_resolve_repo_path(str(gene_info['sequence_path'])))
        gene_info['variants_path'] = str(_resolve_repo_path(str(gene_info['variants_path'])))
    manifest_mode = 'reused_existing_panel_manifest'
else:
    panel_manifest = build_support_ranked_panel_manifest(paths, config)
    manifest_mode = 'rebuilt_panel_manifest'

if TARGET_GENE not in panel_manifest['genes']:
    raise KeyError(f'{TARGET_GENE} not found in support-ranked panel manifest.')

gene_meta = panel_manifest['genes'][TARGET_GENE]
sequence_path = _resolve_repo_path(str(gene_meta['sequence_path']))
if not sequence_path.exists():
    raise FileNotFoundError(f'Missing sequence path for {TARGET_GENE}: {sequence_path}')
sequence = ''.join(line.strip() for line in sequence_path.read_text(encoding='utf-8').splitlines() if not line.startswith('>'))
sequence_length = len(sequence)
expected_scoring_mode = 'full_sequence' if sequence_length <= 1022 else 'local_window'

reference_source_dir = None
if panel_root is not None:
    candidate_reference_dir = panel_root / 'multigene' / TARGET_GENE.lower()
    score_name = f'{TARGET_GENE.lower()}_facebook_esm2_t30_150M_UR50D_scores.csv'
    meta_name = f'{TARGET_GENE.lower()}_facebook_esm2_t30_150M_UR50D_metadata.json'
    if (candidate_reference_dir / score_name).exists() and (candidate_reference_dir / meta_name).exists():
        reference_source_dir = candidate_reference_dir
        reference_target_dir = paths.root / 'esm1v_augmentation' / TARGET_GENE.lower() / 'reference'
        reference_target_dir.mkdir(parents=True, exist_ok=True)
        target_score = reference_target_dir / score_name
        target_meta = reference_target_dir / meta_name
        if not target_score.exists():
            shutil.copy2(candidate_reference_dir / score_name, target_score)
        if not target_meta.exists():
            shutil.copy2(candidate_reference_dir / meta_name, target_meta)

run_context = {
    'target_gene': TARGET_GENE,
    'manifest_mode': manifest_mode,
    'panel_root': str(panel_root) if panel_root is not None else None,
    'reference_cache_reused': reference_source_dir is not None,
    'reference_source_dir': str(reference_source_dir) if reference_source_dir is not None else None,
    'n_total': int(gene_meta['n_total']),
    'n_positive': int(gene_meta['n_positive']),
    'n_negative': int(gene_meta['n_negative']),
    'support_rank': int(gene_meta['support_rank']),
    'sequence_length': sequence_length,
    'expected_scoring_mode': expected_scoring_mode,
    'output_root': str(paths.root),
    'esm1v_model_count': len(ESM1V_MODEL_IDS),
}

display(pd.DataFrame([run_context]))
print(json.dumps(run_context, indent=2))
print('ACABEI , SIGA PARA O PR?XIMO')


In [ ]:
augmentation_summary = run_esm1v_augmentation_suite(
    paths=paths,
    config=config,
    genes=(TARGET_GENE,),
    esm1v_model_ids=ESM1V_MODEL_IDS,
    primary_genes=(TARGET_GENE,),
    run_3b_shadow=RUN_3B_SHADOW,
    panel_manifest=panel_manifest,
)

permutation_summary = run_esm1v_permutation_audit(
    paths=paths,
    config=config,
    genes=(TARGET_GENE,),
    permutation_replicates=PERMUTATION_REPLICATES,
)

gene_result = augmentation_summary['genes'][TARGET_GENE]
perm_result = permutation_summary['genes'][TARGET_GENE]

summary_row = {
    'gene': TARGET_GENE,
    'esm1v_scoring_mode': gene_result['esm1v_scoring_mode'],
    'auc_esm1v_mean': gene_result['auc_esm1v_mean'],
    'auc_augmented_pair_fixed_055': gene_result['auc_augmented_pair_fixed_055'],
    'delta_augmented_vs_esm1v': gene_result['delta_augmented_vs_esm1v'],
    'bootstrap_ci_2p5': gene_result['pair_vs_esm1v_bootstrap_delta']['ci_2p5'],
    'bootstrap_ci_97p5': gene_result['pair_vs_esm1v_bootstrap_delta']['ci_97p5'],
    'bootstrap_p_one_sided_pair_gt_baseline': gene_result['pair_vs_esm1v_bootstrap_delta']['p_one_sided_pair_gt_baseline'],
    'best_alpha_full_surface': gene_result['best_alpha_on_full_surface']['alpha'],
    'best_alpha_full_surface_auc': gene_result['best_alpha_on_full_surface']['auc'],
    'permute_frob_p_ge_observed': perm_result['permute_frob_alignment']['empirical_p_ge_observed'],
    'permute_esm1v_p_ge_observed': perm_result['permute_esm1v_alignment']['empirical_p_ge_observed'],
}

alpha_sweep_df = pd.DataFrame(gene_result['alpha_sweep'])
display(pd.DataFrame([summary_row]))
display(alpha_sweep_df)

decision_summary_path = paths.root / f'{TARGET_GENE.lower()}_h100_decision_summary.json'
decision_summary_path.write_text(json.dumps(summary_row, indent=2) + '\n', encoding='utf-8')

print('Decision summary saved to', decision_summary_path)
print('Gene output directory =', paths.root / 'esm1v_augmentation' / TARGET_GENE.lower())
print('ACABEI , SIGA PARA O PR?XIMO')


## Interpretation rule

Read the result with discipline:
- `GREEN`: positive delta and positive lower CI -> update `abstract` and `content`.
- `AMBER`: positive delta but lower CI still crosses zero -> treat as exploratory only.
- `RED`: non-positive delta -> do not use it as new evidence.

This notebook only attacks the strongest evidence weakness: `BRCA2` looking isolated. It does not fix the whole review by itself.


In [ ]:
delta = float(summary_row['delta_augmented_vs_esm1v'])
ci_2p5 = float(summary_row['bootstrap_ci_2p5'])
best_alpha = float(summary_row['best_alpha_full_surface'])
permute_frob_p = float(summary_row['permute_frob_p_ge_observed'])

if delta > 0.0 and ci_2p5 > 0.0:
    verdict = 'GREEN'
    writing_action = 'Update abstract and content with the new stronger-baseline MSH2 audit.'
elif delta > 0.0 and ci_2p5 <= 0.0:
    verdict = 'AMBER'
    writing_action = 'Keep the core claim conservative; mention MSH2 only as exploratory if needed.'
else:
    verdict = 'RED'
    writing_action = 'Do not use MSH2 as new evidence; only de-jargon the manuscript.'

decision = {
    'target_gene': TARGET_GENE,
    'verdict': verdict,
    'writing_action': writing_action,
    'delta_augmented_vs_esm1v': delta,
    'bootstrap_ci_2p5': ci_2p5,
    'bootstrap_ci_97p5': float(summary_row['bootstrap_ci_97p5']),
    'best_alpha_full_surface': best_alpha,
    'permute_frob_p_ge_observed': permute_frob_p,
}

display(pd.DataFrame([decision]))
print(json.dumps(decision, indent=2))
print('ACABEI , SIGA PARA O PR?XIMO')


In [ ]:
import shutil

bundle_root = paths.root / '_downloads' / f'{TARGET_GENE.lower()}_esm1v_review_recovery'
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

source_dir = paths.root / 'esm1v_augmentation'
if not source_dir.exists():
    raise FileNotFoundError(f'Missing expected output directory: {source_dir}')

shutil.copytree(source_dir, bundle_root / 'esm1v_augmentation', dirs_exist_ok=True)
shutil.copy2(decision_summary_path, bundle_root / decision_summary_path.name)

bundle_manifest = [
    'SpectralBio review recovery stronger-baseline bundle',
    f'target_gene: {TARGET_GENE}',
    f'output_root: {paths.root}',
    f'gene_output_dir: {paths.root / "esm1v_augmentation" / TARGET_GENE.lower()}',
    '',
    'Included:',
    '- esm1v_augmentation/',
    f'- {decision_summary_path.name}',
]
(bundle_root / 'bundle_manifest.txt').write_text('\n'.join(bundle_manifest) + '\n', encoding='utf-8')

archive_path = REPO_ROOT / 'notebooks' / f'{TARGET_GENE.lower()}_esm1v_review_recovery_h100_results.zip'
if archive_path.exists():
    archive_path.unlink()
archive_base = archive_path.with_suffix('')
shutil.make_archive(str(archive_base), 'zip', root_dir=bundle_root.parent, base_dir=bundle_root.name)

print('ZIP ready:', archive_path)
display(FileLink(str(archive_path), result_html_prefix='Download ZIP: '))

if SAVE_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception:
        pass

print('ACABEI , SIGA PARA O PR?XIMO')
